In [84]:
import pandas as pd
import numpy as np
from pathlib import Path

In [85]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Loading the tables

## Joining survey with train service

In [86]:
#service = pd.read_parquet("Data/service_operational_kpis.parquet")

In [87]:
#service.head()

In [88]:
#service.shape

In [89]:
#survey_data = pd.read_parquet("Tables/survey_cleaned.parquet")

In [90]:
#survey_data['metadata_departure_date'] = pd.to_datetime(survey_data['metadata_departure_date']).dt.strftime('%Y%m%d')
#survey_data['service_id'] = survey_data['metadata_train_number'].astype(int).astype(str) + '_' + survey_data['metadata_departure_date']
#df = pd.merge(survey_data, service, on='service_id', how='inner')

In [91]:
df = pd.read_parquet("Data/survey_and_operational_data.parquet")

In [92]:
df.shape

(178833, 211)

In [93]:
df.head()

,metadata_booking_date,metadata_class_of_service,metadata_class_of_service_code,metadata_coach_number,metadata_compensation_flag,metadata_currency,metadata_delay_code,metadata_delay_at_arrival,metadata_departure_date,metadata_departure_hour,metadata_destination_station,metadata_destination_station_code,metadata_disrup,metadata_origin_station,metadata_origin_station_code,metadata_price,metadata_on_board_the_train_v,metadata_overall_experience_v,metadata_q14_op,metadata_q17_op,metadata_recommendation_nps_a,metadata_assistance_service_v,metadata_q_total_duration,metadata_route,metadata_route_code,metadata_seat_number,metadata_segment,metadata_segment_code,metadata_tier_level,metadata_tier_level_code,metadata_train_number,metadata_train_type,metadata_train_type_code,metadata_trips_l12_months,metadata_end_date,metadata_start_date,question_overall_satisfaction_booking_experience,question_overall_satisfaction_wifi_onboard_the_train,question_overall_satisfaction_experience_at_departure_station,question_overall_satisfaction_journey_punctuality,question_overall_satisfaction_comfort_onboard_the_train,question_overall_satisfaction_cleanliness_onboard_the_train,question_overall_satisfaction_overall_service_from_eurostar_staff,question_overall_satisfaction_information_provided_to_you_before_travelling,question_lounge_overall_f_b_your_overall_experience_in_the_premier_lounge,question_lounge_overall_f_b_the_beverage_offering_in_the_lounge,question_lounge_overall_f_b_available_space_seating,question_lounge_overall_f_b_lounge_staff,question_lounge_overall_f_b_cocktail_bar,question_lounge_overall_f_b_newspapers_magazines,question_lounge_overall_f_b_your_welcome_arrival_at_the_lounge,question_lounge_overall_f_b_atmosphere_ambience_in_the_lounge,question_lounge_overall_f_b_furniture_and_d_cor_in_the_lounge,question_lounge_overall_f_b_information_regarding_your_train_departure,question_lounge_overall_f_b_cleanliness_of_the_lounge,question_lounge_overall_f_b_the_food_offering_in_the_lounge,question_lounge_improvement,question_onboard_catering_met_quality_of_food_in_meal_service,question_onboard_catering_met_receiving_preferred_meal_option,question_onboard_catering_met_quality_of_beverages_in_meal_service,question_onboard_catering_met_staff_hospitality_during_meal_service,question_onboard_catering_met_portion_sizes,question_onboard_catering_met_range_of_food_available_in_meal_service,question_onboard_catering_met_range_of_drinks_available_in_meal_service,question_onboard_catering_met_presentation_of_food,question_disruption_performan_the_information_available_to_you,question_disruption_performan_the_compensation_you_received_as_a_result_of_this_disruption,question_disruption_performan_frequency_regularity_of_updates,question_disruption_performan_the_accuracy_of_all_information_provided,question_disruption_performan_the_consistency_of_information_from_different_sources,question_disruption_performan_the_frequency_with_which_you_were_kept_updated,question_disruption_performan_information_provided_in_stations_e_g_from_staff_on_screens,question_disruption_performan_information_on_board_the_train_e_g_from_staff_announcements,question_disruption_performan_information_available_through_the_eurostar_website_or_app,question_disruption_performan_information_received_from_eurostar_via_e_mail_or_sms,question_disruption_performan_passenger_care_hotel_taxi_food_boxes,question_disruption_performan_eurostar_s_overall_management_of_this_disruption,question_usage_hv,question_disability_and_acces,question_assistance_service,question_assistance_service_r,question_language,question_at_the_station_checking_in_to_departures_area,question_at_the_station_boarding_the_train,question_at_the_station_toilets_washrooms_in_the_station,question_at_the_station_the_security_process_baggage_checks_and_scanners,question_at_the_station_passport_control,question_at_the_station_pre_departure_announcements_in_the_station,question_at_the_station_available_space_seating_in_the_waiting_area,qu

## Merging precise Arrival & Departure Delay data

In [94]:
# Load precise delay files
arr_delay = pd.read_csv('Data/Arrival delay.csv')
dep_delay = pd.read_csv('Data/Departure delay.csv')

# Drop last 3 footer rows (Total, NaN, Applied Filters)
arr_delay = arr_delay.drop(arr_delay.tail(3).index)
dep_delay = dep_delay.drop(dep_delay.tail(3).index)

# Correct London timezone bug: St-Pancras arrivals and departures are recorded 60 min too early
arr_delay.loc[arr_delay['Destination Station Name'] == 'St-Pancras-International', 'Arrival Delay (Minutes)'] += 60
dep_delay.loc[dep_delay['Origin Station Name'] == 'St-Pancras-International', 'Departure Delay (Minutes)'] += 60

# Parse date columns (format is dd/mm/yyyy) and build service_id: movementnumber_yyyymmdd
arr_delay['Departure Date'] = pd.to_datetime(arr_delay['Departure Date'], dayfirst=True)
dep_delay['Departure Date'] = pd.to_datetime(dep_delay['Departure Date'], dayfirst=True)

arr_delay['service_id'] = arr_delay['Movement Number'].astype(str) + '_' + arr_delay['Departure Date'].dt.strftime('%Y%m%d')
dep_delay['service_id'] = dep_delay['Movement Number'].astype(str) + '_' + dep_delay['Departure Date'].dt.strftime('%Y%m%d')

print(f'Arrival delay total rows: {len(arr_delay)}')
print(f'Departure delay total rows: {len(dep_delay)}')

Arrival delay total rows: 46259
Departure delay total rows: 52867


In [95]:
def concat_vals(s):
    vals = s.dropna().astype(str).unique()
    return ' | '.join(vals) if len(vals) > 0 else None

cause_cols = ['Cause Category', 'Cause Domain', 'Cause Responsibility']

# For each file: keep first value for all columns, concatenate cause columns across duplicates
arr_agg = {col: 'first' for col in arr_delay.columns if col not in cause_cols + ['service_id']}
arr_agg.update({col: concat_vals for col in cause_cols})
arr_delay = arr_delay.groupby('service_id', sort=False).agg(arr_agg).reset_index()

dep_agg = {col: 'first' for col in dep_delay.columns if col not in cause_cols + ['service_id']}
dep_agg.update({col: concat_vals for col in cause_cols})
dep_delay = dep_delay.groupby('service_id', sort=False).agg(dep_agg).reset_index()


In [96]:
# Left join to keep all services, including those with no recorded delay
df = df.merge(arr_delay, on='service_id', how='left')
df = df.merge(dep_delay, on='service_id', how='left')

print(f'df shape after merging precise delay data: {df.shape}')

df shape after merging precise delay data: (178833, 232)


## Merging Rotations data

In [97]:
# Load rotations file
rotations = pd.read_csv('Data/Rotations.csv')

# Drop last 3 footer rows
rotations = rotations.drop(rotations.tail(3).index)

# Parse date column (format is dd/mm/yyyy) and build service_id: trainnumber_yyyymmdd
rotations['Travel Date'] = pd.to_datetime(rotations['Travel Date'], dayfirst=True)
rotations['service_id'] = rotations['Train Number'].astype(str) + '_' + rotations['Travel Date'].dt.strftime('%Y%m%d')

# Convert HH:MM:SS to decimal minutes (2 decimal places)
def hms_to_minutes(col):
    parts = col.str.split(':', expand=True).astype(float)
    return (parts[0] * 60 + parts[1] + parts[2] / 60).round(2)

for col in ['Theoretical Rotation Time', 'Effective Rotation Time', 'Exceeded Rotation Time']:
    rotations[col] = hms_to_minutes(rotations[col])

# Deduplicate on service_id
rotations = rotations.drop_duplicates('service_id', keep='first')

# Keep only relevant columns
rotations = rotations[['service_id', 'Rame Number', 'Travel Number',
                         'Theoretical Rotation Time', 'Effective Rotation Time', 'Exceeded Rotation Time']]



In [98]:
df = df.merge(rotations, on='service_id', how='left')

## Merging Fleet Availability data

In [99]:
# Load fleet availability
fleet = pd.read_csv('Data/Fleet availability.csv')
fleet['Date'] = pd.to_datetime(fleet['Date'])

# Group by Date + Grouped Parc, sum departure columns
fleet_agg = (
    fleet
    .groupby(['Date', 'Grouped Parc'], as_index=False)
    .agg(
        met_departures=('Sum of Met departures', 'sum'),
        ordered_departures=('Sum of Ordered departures', 'sum'),
        unmet_departures=('Sum of Unmet departures', 'sum')
    )
)

# Fleet Reliability = met / ordered * 100
fleet_agg['fleet_reliability'] = (
    fleet_agg['met_departures'] / fleet_agg['ordered_departures'] * 100
).round(2)

# Rename Grouped Parc categories
fleet_agg['Grouped Parc'] = fleet_agg['Grouped Parc'].replace({
    'Cross-channel': 'Channel',
    'Continent': 'Continental'
})

# Compute totals per date (Continental + Channel combined)
totals = (
    fleet_agg
    .groupby('Date', as_index=False)
    .agg(
        total_met_departures=('met_departures', 'sum'),
        total_ordered_departures=('ordered_departures', 'sum'),
        total_unmet_departures=('unmet_departures', 'sum')
    )
)
totals['total_fleet_reliability'] = (
    totals['total_met_departures'] / totals['total_ordered_departures'] * 100
).round(2)

# Add total columns back to each row (both Continental and Channel get the same totals)
fleet_agg = fleet_agg.merge(totals, on='Date', how='left')

# Format date as yyyymmdd string to match metadata_departure_date
fleet_agg['metadata_departure_date'] = fleet_agg['Date'].dt.strftime('%Y%m%d')


In [100]:
df = df.merge(
    fleet_agg.rename(columns={'Grouped Parc': 'route_type'}),
    on=['metadata_departure_date', 'route_type'],
    how='left'
)
print(f'df shape after merging fleet availability: {df.shape}')

df shape after merging fleet availability: (178833, 246)


In [101]:
df.head(40)

,metadata_booking_date,metadata_class_of_service,metadata_class_of_service_code,metadata_coach_number,metadata_compensation_flag,metadata_currency,metadata_delay_code,metadata_delay_at_arrival,metadata_departure_date,metadata_departure_hour,metadata_destination_station,metadata_destination_station_code,metadata_disrup,metadata_origin_station,metadata_origin_station_code,metadata_price,metadata_on_board_the_train_v,metadata_overall_experience_v,metadata_q14_op,metadata_q17_op,metadata_recommendation_nps_a,metadata_assistance_service_v,metadata_q_total_duration,metadata_route,metadata_route_code,metadata_seat_number,metadata_segment,metadata_segment_code,metadata_tier_level,metadata_tier_level_code,metadata_train_number,metadata_train_type,metadata_train_type_code,metadata_trips_l12_months,metadata_end_date,metadata_start_date,question_overall_satisfaction_booking_experience,question_overall_satisfaction_wifi_onboard_the_train,question_overall_satisfaction_experience_at_departure_station,question_overall_satisfaction_journey_punctuality,question_overall_satisfaction_comfort_onboard_the_train,question_overall_satisfaction_cleanliness_onboard_the_train,question_overall_satisfaction_overall_service_from_eurostar_staff,question_overall_satisfaction_information_provided_to_you_before_travelling,question_lounge_overall_f_b_your_overall_experience_in_the_premier_lounge,question_lounge_overall_f_b_the_beverage_offering_in_the_lounge,question_lounge_overall_f_b_available_space_seating,question_lounge_overall_f_b_lounge_staff,question_lounge_overall_f_b_cocktail_bar,question_lounge_overall_f_b_newspapers_magazines,question_lounge_overall_f_b_your_welcome_arrival_at_the_lounge,question_lounge_overall_f_b_atmosphere_ambience_in_the_lounge,question_lounge_overall_f_b_furniture_and_d_cor_in_the_lounge,question_lounge_overall_f_b_information_regarding_your_train_departure,question_lounge_overall_f_b_cleanliness_of_the_lounge,question_lounge_overall_f_b_the_food_offering_in_the_lounge,question_lounge_improvement,question_onboard_catering_met_quality_of_food_in_meal_service,question_onboard_catering_met_receiving_preferred_meal_option,question_onboard_catering_met_quality_of_beverages_in_meal_service,question_onboard_catering_met_staff_hospitality_during_meal_service,question_onboard_catering_met_portion_sizes,question_onboard_catering_met_range_of_food_available_in_meal_service,question_onboard_catering_met_range_of_drinks_available_in_meal_service,question_onboard_catering_met_presentation_of_food,question_disruption_performan_the_information_available_to_you,question_disruption_performan_the_compensation_you_received_as_a_result_of_this_disruption,question_disruption_performan_frequency_regularity_of_updates,question_disruption_performan_the_accuracy_of_all_information_provided,question_disruption_performan_the_consistency_of_information_from_different_sources,question_disruption_performan_the_frequency_with_which_you_were_kept_updated,question_disruption_performan_information_provided_in_stations_e_g_from_staff_on_screens,question_disruption_performan_information_on_board_the_train_e_g_from_staff_announcements,question_disruption_performan_information_available_through_the_eurostar_website_or_app,question_disruption_performan_information_received_from_eurostar_via_e_mail_or_sms,question_disruption_performan_passenger_care_hotel_taxi_food_boxes,question_disruption_performan_eurostar_s_overall_management_of_this_disruption,question_usage_hv,question_disability_and_acces,question_assistance_service,question_assistance_service_r,question_language,question_at_the_station_checking_in_to_departures_area,question_at_the_station_boarding_the_train,question_at_the_station_toilets_washrooms_in_the_station,question_at_the_station_the_security_process_baggage_checks_and_scanners,question_at_the_station_passport_control,question_at_the_station_pre_departure_announcements_in_the_station,question_at_the_station_available_space_seating_in_the_waiting_area,qu

In [102]:
df.shape

(178833, 246)

In [103]:
df = df.drop(columns=[
    'Date',
    'Actor Name_y',
    'Origin Station Name_y',
    'Destination Station Name_y',
    'Movement Number_y',
    'Departure Date_y',
    'Origin Station Name_x',
    'Destination Station Name_x',
    'Is Station Arrival Service',
    'Is Station Departure Service',
    'Actor Name_x',
    'Movement Number_x',
    'Departure Date_x',
    'Departure Delay (Minutes)_x'
])
print(f'df shape after dropping columns: {df.shape}')

df shape after dropping columns: (178833, 232)


In [104]:
df.head()

,metadata_booking_date,metadata_class_of_service,metadata_class_of_service_code,metadata_coach_number,metadata_compensation_flag,metadata_currency,metadata_delay_code,metadata_delay_at_arrival,metadata_departure_date,metadata_departure_hour,metadata_destination_station,metadata_destination_station_code,metadata_disrup,metadata_origin_station,metadata_origin_station_code,metadata_price,metadata_on_board_the_train_v,metadata_overall_experience_v,metadata_q14_op,metadata_q17_op,metadata_recommendation_nps_a,metadata_assistance_service_v,metadata_q_total_duration,metadata_route,metadata_route_code,metadata_seat_number,metadata_segment,metadata_segment_code,metadata_tier_level,metadata_tier_level_code,metadata_train_number,metadata_train_type,metadata_train_type_code,metadata_trips_l12_months,metadata_end_date,metadata_start_date,question_overall_satisfaction_booking_experience,question_overall_satisfaction_wifi_onboard_the_train,question_overall_satisfaction_experience_at_departure_station,question_overall_satisfaction_journey_punctuality,question_overall_satisfaction_comfort_onboard_the_train,question_overall_satisfaction_cleanliness_onboard_the_train,question_overall_satisfaction_overall_service_from_eurostar_staff,question_overall_satisfaction_information_provided_to_you_before_travelling,question_lounge_overall_f_b_your_overall_experience_in_the_premier_lounge,question_lounge_overall_f_b_the_beverage_offering_in_the_lounge,question_lounge_overall_f_b_available_space_seating,question_lounge_overall_f_b_lounge_staff,question_lounge_overall_f_b_cocktail_bar,question_lounge_overall_f_b_newspapers_magazines,question_lounge_overall_f_b_your_welcome_arrival_at_the_lounge,question_lounge_overall_f_b_atmosphere_ambience_in_the_lounge,question_lounge_overall_f_b_furniture_and_d_cor_in_the_lounge,question_lounge_overall_f_b_information_regarding_your_train_departure,question_lounge_overall_f_b_cleanliness_of_the_lounge,question_lounge_overall_f_b_the_food_offering_in_the_lounge,question_lounge_improvement,question_onboard_catering_met_quality_of_food_in_meal_service,question_onboard_catering_met_receiving_preferred_meal_option,question_onboard_catering_met_quality_of_beverages_in_meal_service,question_onboard_catering_met_staff_hospitality_during_meal_service,question_onboard_catering_met_portion_sizes,question_onboard_catering_met_range_of_food_available_in_meal_service,question_onboard_catering_met_range_of_drinks_available_in_meal_service,question_onboard_catering_met_presentation_of_food,question_disruption_performan_the_information_available_to_you,question_disruption_performan_the_compensation_you_received_as_a_result_of_this_disruption,question_disruption_performan_frequency_regularity_of_updates,question_disruption_performan_the_accuracy_of_all_information_provided,question_disruption_performan_the_consistency_of_information_from_different_sources,question_disruption_performan_the_frequency_with_which_you_were_kept_updated,question_disruption_performan_information_provided_in_stations_e_g_from_staff_on_screens,question_disruption_performan_information_on_board_the_train_e_g_from_staff_announcements,question_disruption_performan_information_available_through_the_eurostar_website_or_app,question_disruption_performan_information_received_from_eurostar_via_e_mail_or_sms,question_disruption_performan_passenger_care_hotel_taxi_food_boxes,question_disruption_performan_eurostar_s_overall_management_of_this_disruption,question_usage_hv,question_disability_and_acces,question_assistance_service,question_assistance_service_r,question_language,question_at_the_station_checking_in_to_departures_area,question_at_the_station_boarding_the_train,question_at_the_station_toilets_washrooms_in_the_station,question_at_the_station_the_security_process_baggage_checks_and_scanners,question_at_the_station_passport_control,question_at_the_station_pre_departure_announcements_in_the_station,question_at_the_station_available_space_seating_in_the_waiting_area,qu

In [105]:
df = df.rename(columns={
    'Arrival Delay (Minutes)': 'Arrival Delay',
    'Departure Delay (Minutes)_y': 'Departure Delay',
    'metadata_recommendation_nps_a': 'metadata_nps'
})

In [106]:
counts = df.groupby('service_id').size()
print(f'service_ids with 1 row:  {(counts == 1).sum()}')
print(f'service_ids with >1 row: {(counts > 1).sum()}')
print(f'\nFull distribution:')
print(counts.value_counts().sort_index())

service_ids with 1 row:  9353
service_ids with >1 row: 36511

Full distribution:
1     9353
2     8910
3     7343
4     5641
5     4161
6     3103
7     2227
8     1518
9     1126
10     832
11     531
12     373
13     265
14     141
15     111
16      92
17      49
18      34
19      18
20      12
21       9
22       7
23       3
24       3
33       1
40       1
Name: count, dtype: int64


In [107]:
df = df[df.groupby('service_id')['service_id'].transform('count') > 1]
print(f'df shape after removing service_ids with only 1 row: {df.shape}')

df shape after removing service_ids with only 1 row: (169480, 232)


In [108]:
df['Aggregated Delay'] = df['Departure Delay'] + df['Arrival Delay']

In [109]:
def nps_category(score):
    if score >= 9:
        return 'promoter'
    elif score >= 7:
        return 'neutral'
    else:
        return 'detractor'

df['nps_category'] = df['metadata_nps'].apply(nps_category)

df['Aggregated NPS'] = df.groupby('service_id')['nps_category'].transform(
    lambda x: round((x == 'promoter').sum() / len(x) * 100 - (x == 'detractor').sum() / len(x) * 100, 2))

In [110]:
df.head(10)

,metadata_booking_date,metadata_class_of_service,metadata_class_of_service_code,metadata_coach_number,metadata_compensation_flag,metadata_currency,metadata_delay_code,metadata_delay_at_arrival,metadata_departure_date,metadata_departure_hour,metadata_destination_station,metadata_destination_station_code,metadata_disrup,metadata_origin_station,metadata_origin_station_code,metadata_price,metadata_on_board_the_train_v,metadata_overall_experience_v,metadata_q14_op,metadata_q17_op,metadata_nps,metadata_assistance_service_v,metadata_q_total_duration,metadata_route,metadata_route_code,metadata_seat_number,metadata_segment,metadata_segment_code,metadata_tier_level,metadata_tier_level_code,metadata_train_number,metadata_train_type,metadata_train_type_code,metadata_trips_l12_months,metadata_end_date,metadata_start_date,question_overall_satisfaction_booking_experience,question_overall_satisfaction_wifi_onboard_the_train,question_overall_satisfaction_experience_at_departure_station,question_overall_satisfaction_journey_punctuality,question_overall_satisfaction_comfort_onboard_the_train,question_overall_satisfaction_cleanliness_onboard_the_train,question_overall_satisfaction_overall_service_from_eurostar_staff,question_overall_satisfaction_information_provided_to_you_before_travelling,question_lounge_overall_f_b_your_overall_experience_in_the_premier_lounge,question_lounge_overall_f_b_the_beverage_offering_in_the_lounge,question_lounge_overall_f_b_available_space_seating,question_lounge_overall_f_b_lounge_staff,question_lounge_overall_f_b_cocktail_bar,question_lounge_overall_f_b_newspapers_magazines,question_lounge_overall_f_b_your_welcome_arrival_at_the_lounge,question_lounge_overall_f_b_atmosphere_ambience_in_the_lounge,question_lounge_overall_f_b_furniture_and_d_cor_in_the_lounge,question_lounge_overall_f_b_information_regarding_your_train_departure,question_lounge_overall_f_b_cleanliness_of_the_lounge,question_lounge_overall_f_b_the_food_offering_in_the_lounge,question_lounge_improvement,question_onboard_catering_met_quality_of_food_in_meal_service,question_onboard_catering_met_receiving_preferred_meal_option,question_onboard_catering_met_quality_of_beverages_in_meal_service,question_onboard_catering_met_staff_hospitality_during_meal_service,question_onboard_catering_met_portion_sizes,question_onboard_catering_met_range_of_food_available_in_meal_service,question_onboard_catering_met_range_of_drinks_available_in_meal_service,question_onboard_catering_met_presentation_of_food,question_disruption_performan_the_information_available_to_you,question_disruption_performan_the_compensation_you_received_as_a_result_of_this_disruption,question_disruption_performan_frequency_regularity_of_updates,question_disruption_performan_the_accuracy_of_all_information_provided,question_disruption_performan_the_consistency_of_information_from_different_sources,question_disruption_performan_the_frequency_with_which_you_were_kept_updated,question_disruption_performan_information_provided_in_stations_e_g_from_staff_on_screens,question_disruption_performan_information_on_board_the_train_e_g_from_staff_announcements,question_disruption_performan_information_available_through_the_eurostar_website_or_app,question_disruption_performan_information_received_from_eurostar_via_e_mail_or_sms,question_disruption_performan_passenger_care_hotel_taxi_food_boxes,question_disruption_performan_eurostar_s_overall_management_of_this_disruption,question_usage_hv,question_disability_and_acces,question_assistance_service,question_assistance_service_r,question_language,question_at_the_station_checking_in_to_departures_area,question_at_the_station_boarding_the_train,question_at_the_station_toilets_washrooms_in_the_station,question_at_the_station_the_security_process_baggage_checks_and_scanners,question_at_the_station_passport_control,question_at_the_station_pre_departure_announcements_in_the_station,question_at_the_station_available_space_seating_in_the_waiting_area,question_at_the_sta

In [111]:
df['Aggregated NPS'] = df['Aggregated NPS'] / 100
df['fleet_reliability'] = df['fleet_reliability'] / 100
df['total_fleet_reliability'] = df['total_fleet_reliability'] / 100

In [112]:
df.head(10)

,metadata_booking_date,metadata_class_of_service,metadata_class_of_service_code,metadata_coach_number,metadata_compensation_flag,metadata_currency,metadata_delay_code,metadata_delay_at_arrival,metadata_departure_date,metadata_departure_hour,metadata_destination_station,metadata_destination_station_code,metadata_disrup,metadata_origin_station,metadata_origin_station_code,metadata_price,metadata_on_board_the_train_v,metadata_overall_experience_v,metadata_q14_op,metadata_q17_op,metadata_nps,metadata_assistance_service_v,metadata_q_total_duration,metadata_route,metadata_route_code,metadata_seat_number,metadata_segment,metadata_segment_code,metadata_tier_level,metadata_tier_level_code,metadata_train_number,metadata_train_type,metadata_train_type_code,metadata_trips_l12_months,metadata_end_date,metadata_start_date,question_overall_satisfaction_booking_experience,question_overall_satisfaction_wifi_onboard_the_train,question_overall_satisfaction_experience_at_departure_station,question_overall_satisfaction_journey_punctuality,question_overall_satisfaction_comfort_onboard_the_train,question_overall_satisfaction_cleanliness_onboard_the_train,question_overall_satisfaction_overall_service_from_eurostar_staff,question_overall_satisfaction_information_provided_to_you_before_travelling,question_lounge_overall_f_b_your_overall_experience_in_the_premier_lounge,question_lounge_overall_f_b_the_beverage_offering_in_the_lounge,question_lounge_overall_f_b_available_space_seating,question_lounge_overall_f_b_lounge_staff,question_lounge_overall_f_b_cocktail_bar,question_lounge_overall_f_b_newspapers_magazines,question_lounge_overall_f_b_your_welcome_arrival_at_the_lounge,question_lounge_overall_f_b_atmosphere_ambience_in_the_lounge,question_lounge_overall_f_b_furniture_and_d_cor_in_the_lounge,question_lounge_overall_f_b_information_regarding_your_train_departure,question_lounge_overall_f_b_cleanliness_of_the_lounge,question_lounge_overall_f_b_the_food_offering_in_the_lounge,question_lounge_improvement,question_onboard_catering_met_quality_of_food_in_meal_service,question_onboard_catering_met_receiving_preferred_meal_option,question_onboard_catering_met_quality_of_beverages_in_meal_service,question_onboard_catering_met_staff_hospitality_during_meal_service,question_onboard_catering_met_portion_sizes,question_onboard_catering_met_range_of_food_available_in_meal_service,question_onboard_catering_met_range_of_drinks_available_in_meal_service,question_onboard_catering_met_presentation_of_food,question_disruption_performan_the_information_available_to_you,question_disruption_performan_the_compensation_you_received_as_a_result_of_this_disruption,question_disruption_performan_frequency_regularity_of_updates,question_disruption_performan_the_accuracy_of_all_information_provided,question_disruption_performan_the_consistency_of_information_from_different_sources,question_disruption_performan_the_frequency_with_which_you_were_kept_updated,question_disruption_performan_information_provided_in_stations_e_g_from_staff_on_screens,question_disruption_performan_information_on_board_the_train_e_g_from_staff_announcements,question_disruption_performan_information_available_through_the_eurostar_website_or_app,question_disruption_performan_information_received_from_eurostar_via_e_mail_or_sms,question_disruption_performan_passenger_care_hotel_taxi_food_boxes,question_disruption_performan_eurostar_s_overall_management_of_this_disruption,question_usage_hv,question_disability_and_acces,question_assistance_service,question_assistance_service_r,question_language,question_at_the_station_checking_in_to_departures_area,question_at_the_station_boarding_the_train,question_at_the_station_toilets_washrooms_in_the_station,question_at_the_station_the_security_process_baggage_checks_and_scanners,question_at_the_station_passport_control,question_at_the_station_pre_departure_announcements_in_the_station,question_at_the_station_available_space_seating_in_the_waiting_area,question_at_the_sta

In [114]:
df.shape

(169480, 235)

In [113]:
df.to_parquet("Data/final_data.parquet", index=False)